In [1]:
import pandas as pd
from typing import Optional, List
from pydantic import BaseModel, Field
import re

In [3]:
# Load the uploaded Excel file to inspect its content
file_path = '../GEARBOx Ontology for LLM_In use.xlsx'
excel_data = pd.ExcelFile(file_path)

# Display the sheet names to understand the structure of the file
excel_data.sheet_names

['GEARBOx v5',
 'GEARBOx v4',
 'GEARBOx v3',
 'Questions & Comments',
 'GEARBOx v2-Kai',
 'GEARBOx v2',
 'GEARBOx v1',
 'Sheet1',
 'Sheet2',
 'Sheet3',
 'xx-GEARBOx',
 'xx-Example']

In [4]:
# Load the sheet to inspect its content
# gearbox_v4_df = excel_data.parse('GEARBOx v4')
gearbox_v4_df = excel_data.parse('GEARBOx v5')
excel_data.close
# Display the first few rows of the dataframe to understand its structure
gearbox_v4_df.head()

,Parameter Name,Type,Description,Examples,MaxLength,Enum,Dulin's comments,ItemType,ItemDescription,ItemExamples,ItemEnum
0,age_min,integer,Does the eligibility criteria mention the mini...,2,10.0,NaN,NaN,NaN,NaN,NaN,NaN
1,age_max,integer,Does the eligibility criteria mention the maxi...,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,weight_min,number,Does the eligibility criteria mention the mini...,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN
3,weight_max,number,Does the eligibility criteria mention the maxi...,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN
4,sex,string,Does the eligibility criteria mention sexes of...,"""Male"", ""Female""",15.0,"""Male"", ""Female"", ""None""",NaN,NaN,NaN,NaN,NaN


In [35]:
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the file token.json.
SCOPES = ["https://www.googleapis.com/auth/spreadsheets.readonly"]
CREDENTIAL_PATH = "../credentials.json"

creds = None
  # The file token.json stores the user's access and refresh tokens, and is
  # created automatically when the authorization flow completes for the first
  # time.
if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)
# If there are no (valid) credentials available, let the user log in.
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            CREDENTIAL_PATH, SCOPES
        )
        creds = flow.run_local_server(port=0)
    # Save the credentials for the next run
    with open("token.json", "w") as token:
        token.write(creds.to_json())

In [68]:
def read_google_sheet(sheet_id: str, sheet_range: str) -> pd.DataFrame:
    service = build("sheets", "v4", credentials=creds)
    sheet = service.spreadsheets()
    result = sheet.values().get(spreadsheetId=sheet_id, range=sheet_range).execute()
    values = result.get("values", [])
    return pd.DataFrame(values[1:], columns=values[0])

# Load the sheet to inspect its content
GOOGLE_SPREADSHEET_ID = "1hpPyjNbRXZFSSo6YwOYaDEZRhE43SROI6A6fs3rgoXc"
GOOGLE_RANGE_NAME = "'GEARBOx v5'!A1:F113"

gearbox_v5_df = read_google_sheet(GOOGLE_SPREADSHEET_ID, GOOGLE_RANGE_NAME)
gearbox_v5_df.Type.str.lower()
# gearbox_v5_df['Examples'] = gearbox_v5_df['Examples'].apply(str)
gearbox_v5_df.head()

,Parameter Name,Type,Description,Examples,MaxLength,Enum
0,age_min,integer,Does the eligibility criteria mention the mini...,2,10,None
1,age_max,integer,Does the eligibility criteria mention the maxi...,25,None,None
2,weight_min,number,Does the eligibility criteria mention the mini...,,10,None
3,weight_max,number,Does the eligibility criteria mention the maxi...,,10,None
4,sex,string,Does the eligibility criteria mention sexes of...,"""Male"", ""Female""",15,"""Male"", ""Female"", ""None"""


In [71]:
# Function to generate class fields and enums
def generate_class_fields_and_enums(df, verb):
    fields = {}
    enum_classes = {}  # Dictionary to store generated enum classes as strings

    for _, row in df.iterrows():
        param_name = row['Parameter Name']
        param_name_exist = f"{param_name}_exist"

        # Determine the field type based on 'Type' and 'Enum' columns
        if row['Type'] in ['integer', 'number']:
            field_type = 'Optional[int]'
        elif pd.notna(row['Enum']) and 'Yes' in row['Enum']:
            field_type = 'Optional[Yes_no]'  # Assuming YesOrNo is a defined type elsewhere
        elif pd.notna(row['Enum']):
            field_type = f"Optional[List[{param_name.capitalize()}]]"
            
            # Create an enum class for this parameter
            enum_items = [item.strip().strip('"') for item in row['Enum'].split(',')]
            enum_class_str = f"class {param_name.capitalize()}(Enum):\n"
            for item in enum_items:
                if "(" in item and ")" in item:
                    enum_name = re.search(r"\((.*?)\)", item).group(1)
                    # enum_name = enum_name.strip().replace(" ", "").replace("-", "").upper()
                    enum_name = enum_name.strip().replace(" ", "_").replace("-", "_").upper()
                else:
                    # enum_name = item.strip().replace("-", "").replace(" ", "").upper()
                    enum_name = item.strip().replace("-", "_").replace(" ", "_").upper()
                enum_class_str += f"    {enum_name} = \"{item}\"\n"
            
            enum_classes[param_name.capitalize()] = enum_class_str
            
        else:
            field_type = 'Optional[str]'

        # Combine description and examples if available
        description = row['Description']

        print((row['Examples'].strip() != ''), row['Examples'])
        if row['Examples'].strip() != '':
            example = row['Examples'].replace('"', "'")
            description += f" Example: {example}"

        first_yesno, second_criteria = description.split("If so,", 1)
        first_yesno = first_yesno.strip()
        second_criteria = second_criteria.strip().replace("{verb}", verb)

        fields[param_name_exist] = f"Yes_no = Field(..., description=\"{first_yesno}\")"
        fields[param_name] = f"{field_type} = Field(\n        None, \n        description=\"{second_criteria}\")"
    
    enum_items = ["Yes","No"]
    enum_class_str = f"class Yes_no(Enum):\n"
    for item in enum_items:
        enum_name = item.upper()
        enum_class_str += f"    {enum_name} = \"{item}\"\n"
    enum_classes["Yes_no"] = enum_class_str

    return fields, enum_classes


# Generate class fields from the DataFrame
class_fields_ic, enum_classes = generate_class_fields_and_enums(gearbox_v5_df, "included")
class_fields_ec, _ = generate_class_fields_and_enums(gearbox_v5_df, "excluded")

class_definition = "from enum import Enum\nfrom pydantic import BaseModel, Field\nfrom typing import Optional, List\n\n"
for enum_name, enum_definition in enum_classes.items():
    class_definition += f"{enum_definition}\n"

# Generating the ic class as a string
class_definition_ic = class_definition + "class Entity(BaseModel):\n"
for field_name, field_definition in class_fields_ic.items():
    class_definition_ic += f"    {field_name}: {field_definition}\n"

# Generating the ec class as a string
class_definition_ec = class_definition + "class Entity(BaseModel):\n"
for field_name, field_definition in class_fields_ec.items():
    class_definition_ec += f"    {field_name}: {field_definition}\n"

True 2
True 25
False 
False 
True "Male", "Female"
True Diagnoses include "Acute promyelocytic leukemia (APL)", "Acute myeloid leukemia (AML)", "Down syndrome acute myeloid leukemia (DS AML)", "B-cell acute lymphoblastic leukemia (B-ALL)", "Juvenile myelomonocytic leukemia (JMML)", "Mixed-phenotype acute leukemia (MPAL)", "Myelodysplastic syndromes (MDS)", "Myelodysplastic syndrome who have progressed to AML (MDS-AML)", "Treatment-related acute myeloid leukemia (tAML)", "T-cell acute lymphoblastic leukemia (T-ALL)", "Acute lymphoblastic leukemia (ALL)", "KMT2A acute lymphoblastic leukemia", "Early T-cell precursor acute lymphoblastic leukemia (ETP-ALL)", "Ambiguous lineage acute leukemia (ALAL)" 
True refractory disease are diseases such as "Refractory leukemia"
False 
True the answer is 1 if "refractory to at least one course of salvage therapy", the answer is 2 if "persistent after at least two courses of induction "chemotherapy", the answer is 2 if "refractory disease after >= 2 cyc

In [72]:
# Define the file path to save the class definition
output_IC_file_path = "Entity_IC.py"
output_EC_file_path = "Entity_EC.py"

# Save the class definition to a file
with open(output_IC_file_path, 'w') as file:
    file.write(class_definition_ic)

# Save the class definition to a file
with open(output_EC_file_path, 'w') as file:
    file.write(class_definition_ec)